# 34. Topology (TP) — Data Management

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Maps-and-Spatial-Analyses/blob/6949a8bf2f56c2e0b73a4da537beaf686cd00452/notebooks/34_TP_topology.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `notebooks/34_TP_topology.ipynb` at `6949a8bf2f56c2e0b73a4da537beaf686cd00452`; original SHA-256 `c044cffde4d56739f625b695c183d63fbb24ce2cc7d01e1800450d2a489cd789`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 34. Topology (TP) — Data Management

[Index](https://github.com/jltobias/JupyterLite-Maps-and-Spatial-Analyses/blob/6949a8bf2f56c2e0b73a4da537beaf686cd00452/notebooks/34_TP_topology.ipynb) | [Previous](33_AD_address-geocoding.ipynb) | [Next](35_LR_linear-referencing.ipynb)

**Concept focus:** Finds geometry errors such as gaps, overlaps, and dangling lines.

**Map-first learning goal:** use real latitude/longitude around **Atlanta, Georgia** to explore topology rules. The values are synthetic, but the coordinates, basemap, and geographic context are real.

Each notebook produces several interactive maps:

1. **Observation map** — synthetic point measurements on a real OpenStreetMap basemap.
2. **Analysis map** — concept-specific derived layer such as buffers, zones, grids, routes, or surfaces.
3. **Cognition map** — combined evidence layer for interpretation, comparison, and decision-making.

> Teaching note: these examples avoid heavy native geospatial libraries so they run reliably in JupyterLite/Pyodide. They are intentionally small, inspectable, and remixable.


In [ ]:
# Browser-friendly interactive maps for JupyterLite.
# The maps use real-world latitude/longitude coordinates and OpenStreetMap tiles.
# No GeoPandas, GDAL, or server-side GIS stack is required.
import json
import math
import random
from html import escape
from IPython.display import HTML, display

random.seed(7)


def make_observations(center_lat, center_lon, n=28, spread=0.045, theme="measurement"):
    """Create synthetic point observations around a real-world location."""
    observations = []
    for i in range(n):
        # Slightly elliptical cloud so spatial structure is visible on a map.
        lat = center_lat + random.uniform(-spread, spread)
        lon = center_lon + random.uniform(-spread * 1.25, spread * 1.25)
        distance = math.hypot((lat-center_lat) * 111, (lon-center_lon) * 85)
        value = round(55 + 30 * math.exp(-distance / 4) + random.uniform(-8, 8), 1)
        observations.append({
            "id": f"P{i+1:02d}",
            "lat": round(lat, 6),
            "lon": round(lon, 6),
            "value": value,
            "priority": "high" if value >= 72 else "medium" if value >= 60 else "low",
            "theme": theme,
        })
    return observations


def make_grid(center_lat, center_lon, rows=5, cols=5, spacing=0.025):
    """Create a simple lat/lon cell grid represented as GeoJSON polygons."""
    features = []
    start_lat = center_lat - rows * spacing / 2
    start_lon = center_lon - cols * spacing / 2
    for r in range(rows):
        for c in range(cols):
            south = start_lat + r * spacing
            west = start_lon + c * spacing
            north = south + spacing
            east = west + spacing
            # Synthetic surface value with a spatial gradient.
            value = round(30 + 8*r + 5*c + 10*math.sin((r+c)/2), 1)
            features.append({
                "type": "Feature",
                "properties": {"cell": f"R{r+1}C{c+1}", "value": value},
                "geometry": {"type": "Polygon", "coordinates": [[
                    [west, south], [east, south], [east, north], [west, north], [west, south]
                ]]}
            })
    return {"type": "FeatureCollection", "features": features}


def make_zones(center_lat, center_lon):
    """Create three synthetic neighborhood polygons around a real place."""
    offsets = [(-0.045, -0.055, -0.005, 0.005), (-0.005, -0.03, 0.04, 0.035), (-0.04, 0.015, 0.025, 0.065)]
    names = ["West zone", "Central zone", "East zone"]
    features = []
    for name, (s, w, n, e) in zip(names, offsets):
        features.append({
            "type": "Feature",
            "properties": {"name": name},
            "geometry": {"type": "Polygon", "coordinates": [[
                [center_lon+w, center_lat+s], [center_lon+e, center_lat+s],
                [center_lon+e, center_lat+n], [center_lon+w, center_lat+n],
                [center_lon+w, center_lat+s]
            ]]}
        })
    return {"type": "FeatureCollection", "features": features}


def make_route(center_lat, center_lon):
    """Create a synthetic route polyline in real-world coordinates."""
    coords = []
    for i in range(9):
        coords.append([center_lat - 0.05 + i*0.012, center_lon - 0.055 + math.sin(i/1.5)*0.035 + i*0.006])
    return coords


def leaflet_iframe(title, center, layers, height=560, zoom=12):
    """Display an interactive Leaflet map in a sandboxed iframe."""
    payload = json.dumps({"title": title, "center": center, "zoom": zoom, "layers": layers})
    inner = f"""
<!doctype html>
<html>
<head>
<meta charset='utf-8'>
<meta name='viewport' content='width=device-width, initial-scale=1'>
<link rel='stylesheet' href='https://unpkg.com/leaflet@1.9.4/dist/leaflet.css'>
<script src='https://unpkg.com/leaflet@1.9.4/dist/leaflet.js'></script>
<style>
  html, body, #map {{ height: 100%; margin: 0; }}
  .legend {{ background: white; padding: 8px 10px; border-radius: 8px; box-shadow: 0 1px 5px #999; font: 13px/1.4 Arial, sans-serif; }}
  .legend b {{ display:block; margin-bottom:4px; }}
</style>
</head>
<body>
<div id='map'></div>
<script>
const spec = {payload};
const map = L.map('map').setView(spec.center, spec.zoom);
L.tileLayer('https://tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{
  maxZoom: 19,
  attribution: '&copy; OpenStreetMap contributors'
}}).addTo(map);
const overlays = {{}};
function colorForValue(v) {{
  return v >= 75 ? '#7f0000' : v >= 65 ? '#d7301f' : v >= 55 ? '#fc8d59' : '#91bfdb';
}}
function styleFeature(feature) {{
  const v = feature.properties && feature.properties.value || 50;
  return {{color: colorForValue(v), weight: 2, fillColor: colorForValue(v), fillOpacity: 0.28}};
}}
for (const layer of spec.layers) {{
  let group = L.layerGroup();
  if (layer.type === 'points') {{
    for (const p of layer.data) {{
      const marker = L.circleMarker([p.lat, p.lon], {{
        radius: 5 + Math.max(0, (p.value || 50)-45)/8,
        color: colorForValue(p.value || 50), fillColor: colorForValue(p.value || 50),
        fillOpacity: 0.75, weight: 1
      }}).bindPopup(`<b>${{p.id}}</b><br>Value: ${{p.value}}<br>Priority: ${{p.priority || 'n/a'}}`);
      marker.addTo(group);
    }}
  }}
  if (layer.type === 'geojson') {{
    L.geoJSON(layer.data, {{
      style: styleFeature,
      onEachFeature: function(feature, lyr) {{
        const props = feature.properties || {{}};
        lyr.bindPopup(Object.entries(props).map(([k,v]) => `<b>${{k}}</b>: ${{v}}`).join('<br>'));
      }}
    }}).addTo(group);
  }}
  if (layer.type === 'route') {{
    L.polyline(layer.data, {{color: layer.color || '#08519c', weight: 5, opacity: 0.8}}).addTo(group)
      .bindPopup(layer.popup || 'Route');
    for (const [idx, pt] of layer.data.entries()) {{
      if (idx === 0 || idx === layer.data.length-1) L.marker(pt).addTo(group).bindPopup(idx === 0 ? 'Start' : 'End');
    }}
  }}
  if (layer.type === 'buffers') {{
    for (const p of layer.data) {{
      L.circle([p.lat, p.lon], {{radius: layer.radius_m || 1200, color: '#756bb1', fillColor: '#bcbddc', fillOpacity: 0.18}}).addTo(group)
        .bindPopup(`Buffer around ${{p.id}}`);
    }}
  }}
  group.addTo(map);
  overlays[layer.name] = group;
}}
L.control.layers(null, overlays, {{collapsed: false}}).addTo(map);
const legend = L.control({{position: 'bottomright'}});
legend.onAdd = function() {{
  const div = L.DomUtil.create('div', 'legend');
  div.innerHTML = `<b>${{spec.title}}</b><span style='color:#7f0000'>●</span> high &nbsp; <span style='color:#fc8d59'>●</span> medium &nbsp; <span style='color:#91bfdb'>●</span> low<br>Toggle layers to compare inputs and outputs.`;
  return div;
}};
legend.addTo(map);
</script>
</body>
</html>"""
    framed = "<iframe srcdoc=\"{}\" width=\"100%\" height=\"{}\" style=\"border:1px solid #ccc; border-radius:10px;\"></iframe>".format(
        escape(inner, quote=True), height
    )
    display(HTML(framed))

In [ ]:
CENTER = [33.749000, -84.388000]
PLACE = 'Atlanta, Georgia'
ELEMENT = 'Topology'
CATEGORY = 'Data Management'
CONCEPT_HINT = 'topology rules'

observations = make_observations(CENTER[0], CENTER[1], n=30, spread=0.045, theme=ELEMENT)
zones = make_zones(CENTER[0], CENTER[1])
grid = make_grid(CENTER[0], CENTER[1], rows=5, cols=5, spacing=0.025)
route = make_route(CENTER[0], CENTER[1])

print(f'{ELEMENT} notebook centered on {PLACE} at {CENTER}')
print(f'Created {len(observations)} synthetic observations, {len(zones["features"])} zones, {len(grid["features"])} grid cells, and one route.')


## Map 1 — Observations in real geographic context

In [ ]:
leaflet_iframe(
    title=f"{ELEMENT}: observations in {PLACE}",
    center=CENTER,
    layers=[
        {"name": "Synthetic observations", "type": "points", "data": observations},
        {"name": "Reference zones", "type": "geojson", "data": zones},
    ],
    zoom=12,
)


## Map 2 — Concept-specific analysis layer

In [ ]:
# Attribute/data-management analysis: join tabular attributes to mapped coordinates.
# Here, high-priority observations are mapped with zones to show how table fields become spatial evidence.
high_priority = [p for p in observations if p['priority'] == 'high']
leaflet_iframe(
    title=f"{ELEMENT}: table attributes become map evidence",
    center=CENTER,
    layers=[
        {"name": "All records", "type": "points", "data": observations},
        {"name": "High-priority records", "type": "points", "data": high_priority},
        {"name": "Management zones", "type": "geojson", "data": zones},
    ],
    zoom=12,
)


## Map 3 — Combined interpretation map

In [ ]:
# Cognition map: combine the main inputs and derived layers.
# Ask: Where do the observations, zones, route/access, and surface evidence agree or disagree?
top_points = sorted(observations, key=lambda p: p['value'], reverse=True)[:8]
leaflet_iframe(
    title=f"{ELEMENT}: combined interpretation map",
    center=CENTER,
    layers=[
        {"name": "Top observations", "type": "points", "data": top_points},
        {"name": "Reference zones", "type": "geojson", "data": zones},
        {"name": "Synthetic surface/grid", "type": "geojson", "data": grid},
        {"name": "Movement/access path", "type": "route", "data": route, "popup": "Reference route"},
    ],
    zoom=12,
)


## Reflection prompts

- Which mapped layer best explains **Topology** in this example: points, zones, grid cells, buffers, or paths?
- What changes when you zoom out from the neighborhood scale to the city scale?
- Which variables would need to be real observations before this could support an operational decision?
- How could you adapt this notebook for a CDC/HHS public-health example while preserving privacy?

## Try changing the model

Edit `spread`, `n`, `rows`, `cols`, or the buffer radius and rerun the maps. Good geospatial learning comes from seeing how spatial conclusions respond to different assumptions.
